# Merchant Pricing and Profitability

A checked synthetic example of how transaction mix and pricing assumptions affect contribution margin.

## 1. Load the merchant-month data

The data is authored and deterministic. Each row is one fictional merchant for one representative month.

In [1]:
from pathlib import Path

import pandas as pd

from src.analyze_pricing import calculate_merchant_economics
from src.analyze_pricing import compare_plans

ROOT = Path.cwd()
merchants = pd.read_csv(ROOT / 'data' / 'synthetic_merchants.csv')
print(merchants.shape)
print(merchants[['monthly_sales_cad', 'transactions', 'refund_rate']].describe().round(2).to_string())

(360, 10)
       monthly_sales_cad  transactions  refund_rate
count             360.00        360.00       360.00
mean            52144.22        876.73         0.03
std             84725.91       1538.17         0.02
min              1916.59         26.00         0.00
25%              8575.26        133.00         0.01
50%             21008.23        293.50         0.02
75%             58283.03        943.50         0.04
max            619181.76      11105.00         0.14


## 2. Check the row grain and totals

The merchant identifier should be unique and the transaction and sales totals should be positive.

In [2]:
checks = pd.DataFrame({
    'check': ['unique merchant rows', 'positive monthly sales', 'positive transaction counts'],
    'passed': [merchants['merchant_id'].is_unique, merchants['monthly_sales_cad'].gt(0).all(), merchants['transactions'].gt(0).all()],
    'observed': [merchants['merchant_id'].nunique(), round(merchants['monthly_sales_cad'].sum(), 2), int(merchants['transactions'].sum())]
})
print(checks.to_string(index=False))

                      check  passed    observed
       unique merchant rows    True      360.00
     positive monthly sales    True 18771918.06
positive transaction counts    True   315622.00


## 3. Compare the three pricing options

Every plan uses the same synthetic underlying payment costs. Only the markup, transaction fee and monthly fee change.

In [3]:
detail, summary = compare_plans(merchants)
print(summary.to_string(index=False))

   plan  merchants   sales_cad  transactions  fee_revenue_cad  total_cost_cad  contribution_margin_cad  weighted_effective_rate_pct  contribution_margin_rate_pct  negative_margin_merchants
Current        360 18771918.06        315622        385509.56       241746.46                143763.10                       2.0537                       37.2917                          0
 Growth        360 18771918.06        315622        359036.43       241746.46                117289.98                       1.9126                       32.6680                          0
 Simple        360 18771918.06        315622        373149.88       241746.46                131403.43                       1.9878                       35.2147                          0


## 4. Compare merchant segments

A weighted effective rate uses total fee revenue divided by total processed sales. This avoids averaging merchant percentages equally.

In [4]:
segment = detail.groupby(['plan', 'segment'], as_index=False).agg(sales_cad=('scenario_sales_cad', 'sum'), fee_revenue_cad=('fee_revenue_cad', 'sum'), contribution_margin_cad=('contribution_margin_cad', 'sum'))
segment['weighted_effective_rate_pct'] = segment['fee_revenue_cad'] / segment['sales_cad'] * 100
print(segment.round(2).to_string(index=False))

   plan    segment  sales_cad  fee_revenue_cad  contribution_margin_cad  weighted_effective_rate_pct
Current Enterprise 8235766.89        163061.33                 60921.67                         1.98
Current      Micro  710137.07         15952.04                  7190.59                         2.25
Current Mid-market 7462201.17        156381.22                 56272.63                         2.10
Current      Small 2363812.93         50114.97                 19378.22                         2.12
 Growth Enterprise 8235766.89        149259.09                 47119.43                         1.81
 Growth      Micro  710137.07         16543.23                  7781.77                         2.33
 Growth Mid-market 7462201.17        145231.06                 45122.47                         1.95
 Growth      Small 2363812.93         48003.05                 17266.31                         2.03
 Simple Enterprise 8235766.89        159790.03                 57650.37                    

## 5. Test a cost shock

Pricing is held fixed while credit costs rise by 100 basis points. This is a stress test, not a forecast.

In [5]:
stress = calculate_merchant_economics(merchants, 'Growth', cost_shock_bps=100)
stress_summary = pd.DataFrame({
    'measure': ['contribution margin', 'negative-margin merchants'],
    'observed': [round(stress['contribution_margin_cad'].sum(), 2), int(stress['contribution_margin_cad'].lt(0).sum())]
})
print(stress_summary.to_string(index=False))

                  measure  observed
      contribution margin -13632.02
negative-margin merchants    143.00


## 6. Inspect break-even pressure

The break-even markup is the additional volume rate needed after the selected fixed and transaction fees. A negative value means those fixed components already cover the modelled costs.

In [6]:
current = detail[detail['plan'].eq('Current')]
print(current[['merchant_id', 'segment', 'channel', 'scenario_sales_cad', 'contribution_margin_cad', 'break_even_markup_bps']].nsmallest(8, 'contribution_margin_cad').round(2).to_string(index=False))

merchant_id segment   channel  scenario_sales_cad  contribution_margin_cad  break_even_markup_bps
      M0033   Micro In person             1916.59                    31.46                 -24.98
      M0147   Micro     Mixed             2527.43                    36.17                  59.69
      M0245   Micro     Mixed             2654.95                    36.37                  41.61
      M0202   Micro In person             2156.71                    37.53                  15.69
      M0320   Micro In person             2681.93                    38.78                  47.67
      M0327   Micro In person             2635.65                    39.41                  17.43
      M0355   Micro     Mixed             2995.70                    39.70                  79.80
      M0167   Micro In person             3044.16                    39.71                  57.16


## 7. Interpretation

The current plan produces the largest total contribution margin in the base case. The growth plan lowers the portfolio weighted effective rate, but a large unpriced cost shock can turn the portfolio negative. The result supports a monitored pricing decision, not a claim about a real provider or merchant.